# RAG Pipeline - Complete Implementation

This notebook implements a complete RAG pipeline with:
- Document ingestion (TXT and PDF files)
- Text chunking
- Embedding generation
- Vector store creation (ChromaDB)
- Semantic search retrieval


In [ ]:
# ================================
# Import Required Libraries
# ================================
import os
import shutil
from pathlib import Path

from langchain_community.document_loaders import (
    TextLoader,
    DirectoryLoader,
    PyPDFLoader
)
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

print("All libraries imported successfully!")


All libraries imported successfully!


In [ ]:
# ================================
# Step 1: Clean Existing Vector Store
# ================================
# Delete existing ChromaDB directory to start fresh
chroma_db_path = Path("../data/chroma_db")

if chroma_db_path.exists():
    print(f"Deleting existing ChromaDB at: {chroma_db_path}")
    shutil.rmtree(chroma_db_path)
    print("Existing ChromaDB deleted successfully!")
else:
    print(f"ChromaDB directory does not exist at: {chroma_db_path}")
    print("Proceeding with fresh vector store creation...")

print("\n" + "="*50)

Deleting existing ChromaDB at: ../data/chroma_db
Existing ChromaDB deleted successfully!



In [ ]:
# ================================
# Step 2: Data Ingestion
# ================================
# Load both TXT and PDF files

# Define paths
text_files_path = "../data/text_files"
pdf_files_path = "../data/pdf"

# -------------------------------
# Load Text Files
# -------------------------------
print("Loading Text Files...")
print(f"Text files directory: {text_files_path}")

if os.path.exists(text_files_path):
    print(f"Files found: {os.listdir(text_files_path)}")
    
    text_loader = DirectoryLoader(
        text_files_path,
        glob="*.txt",
        loader_cls=TextLoader,
        show_progress=True
    )
    
    text_documents = text_loader.load()
    print(f"✓ Loaded {len(text_documents)} text documents")
else:
    print(f"✗ Directory not found: {text_files_path}")
    text_documents = []

# -------------------------------
# Load PDF Files
# -------------------------------
print("\nLoading PDF Files...")
print(f"PDF files directory: {pdf_files_path}")

if os.path.exists(pdf_files_path):
    print(f"Files found: {os.listdir(pdf_files_path)}")
    
    pdf_loader = DirectoryLoader(
        pdf_files_path,
        glob="*.pdf",
        loader_cls=PyPDFLoader,
        show_progress=True
    )
    
    pdf_documents = pdf_loader.load()
    print(f"✓ Loaded {len(pdf_documents)} PDF documents")
else:
    print(f"✗ Directory not found: {pdf_files_path}")
    pdf_documents = []

# -------------------------------
# Combine All Documents
# -------------------------------
documents = text_documents + pdf_documents

print("\n" + "="*50)
print(f"TOTAL DOCUMENTS LOADED: {len(documents)}")
print("="*50)

# Verify documents have required structure
if documents:
    print(f"\n✓ All documents have 'page_content' and 'metadata'")
    print(f"\nSample document:")
    print(f"  Source: {documents[0].metadata.get('source', 'N/A')}")
    print(f"  Page: {documents[0].metadata.get('page', 'N/A')}")
    print(f"  Content length: {len(documents[0].page_content)} characters")
    print(f"  Content preview: {documents[0].page_content[:200]}...")
else:
    raise ValueError("No documents were loaded! Please check file paths.")


Loading Text Files...
Text files directory: ../data/text_files
Files found: ['python_intro.txt', 'simple_text_1.txt', 'simple_text_3.txt', 'simple_text_2.txt', 'machine_learning.txt']


100%|██████████| 5/5 [00:00<00:00, 750.73it/s]


✓ Loaded 5 text documents

Loading PDF Files...
PDF files directory: ../data/pdf
Files found: ['7_IR_Practice Questions_26.8.25.pdf', 'Fast Learner Assignment .pdf', '1_IT4154 IR_Assign_11.9.25.pdf', 'IT4154_IR_PE5_24.9.25_Sol.pdf', '2_IT4154_IR_PE5_Resess_Sol.pdf', '11_Eval_Practice Questions_3.9.25.pdf', '14_IR_Chpt9_Practice Ques_11.9.25.pdf']


100%|██████████| 7/7 [00:00<00:00, 17.69it/s]

✓ Loaded 37 PDF documents

TOTAL DOCUMENTS LOADED: 42

✓ All documents have 'page_content' and 'metadata'

Sample document:
  Source: ../data/text_files/python_intro.txt
  Page: N/A
  Content length: 683 characters
  Content preview: Python Introduction

Python is a high-level, interpreted programming language known for its simplicity and readability. It was created by Guido van Rossum and first released in 1991.

Key Features:
- ...


In [ ]:
# ================================
# Step 3: Text Chunking
# ================================
# Split documents into chunks for embedding

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=900,      # Target chunk size (800-1000 range)
    chunk_overlap=175,   # Overlap between chunks (150-200 range)
    length_function=len,
)

# Split all documents into chunks
chunks = text_splitter.split_documents(documents)

print("="*50)
print("CHUNKING RESULTS")
print("="*50)
print(f"Original documents: {len(documents)}")
print(f"Total chunks created: {len(chunks)}")
print(f"Average chunks per document: {len(chunks) / len(documents):.2f}")

# Verify chunks are non-empty
assert len(chunks) > 0, "ERROR: No chunks were created!"

print(f"\n✓ Chunks list is non-empty: {len(chunks)} chunks")

# Show sample chunk
print(f"\nSample chunk:")
print(f"  Length: {len(chunks[0].page_content)} characters")
print(f"  Source: {chunks[0].metadata.get('source', 'N/A')}")
print(f"  Page: {chunks[0].metadata.get('page', 'N/A')}")
print(f"  Content preview: {chunks[0].page_content[:300]}...")


CHUNKING RESULTS
Original documents: 42
Total chunks created: 69
Average chunks per document: 1.64

✓ Chunks list is non-empty: 69 chunks

Sample chunk:
  Length: 678 characters
  Source: ../data/text_files/python_intro.txt
  Page: N/A
  Content preview: Python Introduction

Python is a high-level, interpreted programming language known for its simplicity and readability. It was created by Guido van Rossum and first released in 1991.

Key Features:
- Easy to learn syntax
- Dynamic typing
- Extensive standard library
- Cross-platform compatibility

P...


In [ ]:
# ================================
# Step 4: Initialize Embeddings
# ================================
# Use HuggingFace embeddings model

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={'device': 'cpu'}
)

print("="*50)
print("EMBEDDINGS MODEL")
print("="*50)
print("Model: all-MiniLM-L6-v2")
print("Status: ✓ Loaded successfully")

# Test embedding generation
sample_text = "This is a test sentence for embedding"
sample_embedding = embeddings.embed_query(sample_text)
print(f"Embedding dimension: {len(sample_embedding)}")
print(f"Sample embedding (first 5 values): {sample_embedding[:5]}")


EMBEDDINGS MODEL
Model: all-MiniLM-L6-v2
Status: ✓ Loaded successfully
Embedding dimension: 384
Sample embedding (first 5 values): [0.01976894959807396, 0.010906081646680832, 0.07236586511135101, 0.040319111198186874, 0.05278414487838745]


In [ ]:
# ================================
# Step 5: Create Vector Store (ChromaDB)
# ================================
# Create and persist vector store with all chunks

print("="*50)
print("CREATING VECTOR STORE")
print("="*50)
print(f"Chunks to embed: {len(chunks)}")
print(f"Persist directory: ../data/chroma_db")
print("\nCreating embeddings and vector store (this may take a moment)...")

# Create vector store from documents
# This will automatically generate embeddings and store them
vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="../data/chroma_db"
)

print("✓ Vector store created successfully!")

# Verify embeddings were stored
collection = vector_store._collection
embedding_count = collection.count()

print("\n" + "="*50)
print("VECTOR STORE VERIFICATION")
print("="*50)
print(f"Total embeddings stored: {embedding_count}")

# Critical assertion
assert embedding_count > 0, f"ERROR: Vector store has 0 embeddings! Expected > 0."

print(f"✓ Verification passed: {embedding_count} embeddings stored")
print(f"✓ Vector store persisted to: ../data/chroma_db")


CREATING VECTOR STORE
Chunks to embed: 69
Persist directory: ../data/chroma_db

Creating embeddings and vector store (this may take a moment)...


InternalError: Error updating collection: Database error: error returned from database: (code: 1032) attempt to write a readonly database

In [ ]:
# ================================
# Step 6: Reload Vector Store (Verification)
# ================================
# Reload from disk to verify persistence works correctly

print("="*50)
print("RELOADING VECTOR STORE FROM DISK")
print("="*50)

# Delete the in-memory vector_store to force reload
del vector_store

# Reload from disk
vector_store = Chroma(
    persist_directory="../data/chroma_db",
    embedding_function=embeddings
)

# Verify it loaded correctly
collection = vector_store._collection
embedding_count = collection.count()

print(f"✓ Vector store reloaded successfully")
print(f"✓ Embeddings count: {embedding_count}")
assert embedding_count > 0, "ERROR: Reloaded vector store has 0 embeddings!"


In [ ]:
# ================================
# Step 7: Semantic Search - Query "web crawling system"
# ================================

query = "web crawling system"

print("="*50)
print("SEMANTIC SEARCH")
print("="*50)
print(f"Query: '{query}'")
print(f"\nSearching for relevant documents...\n")

# Perform similarity search
results = vector_store.similarity_search_with_score(query, k=5)

print(f"Found {len(results)} relevant results:\n")

if len(results) == 0:
    print("⚠️  WARNING: No results found!")
    print("This may indicate:")
    print("  - The query doesn't match any content in the documents")
    print("  - The vector store was not created correctly")
else:
    for i, (doc, score) in enumerate(results, 1):
        print(f"{'='*60}")
        print(f"Result {i} (Similarity Score: {score:.4f})")
        print(f"{'='*60}")
        
        # Extract source file name
        source = doc.metadata.get('source', 'N/A')
        source_name = os.path.basename(source) if source != 'N/A' else 'N/A'
        
        # Extract page number
        page = doc.metadata.get('page', 'N/A')
        
        print(f"Source File: {source_name}")
        print(f"Page Number: {page}")
        print(f"\nContent Snippet:")
        print(f"{doc.page_content[:400]}...")
        print()


Embedding model loaded successfully!


In [ ]:
# ================================
# Additional Query Examples
# ================================
# Test with other queries to verify the pipeline works

test_queries = [
    "information retrieval",
    "machine learning",
    "python programming"
]

print("="*50)
print("ADDITIONAL QUERY TESTS")
print("="*50)

for query in test_queries:
    print(f"\nQuery: '{query}'")
    results = vector_store.similarity_search(query, k=2)
    print(f"  Found {len(results)} results")
    if results:
        print(f"  Top result source: {os.path.basename(results[0].metadata.get('source', 'N/A'))}")


Vector store is persisted to: ../data/chroma_db
You can load it later using the code above.


In [ ]:
# ================================
# Pipeline Summary
# ================================
# Verify the complete pipeline is working

print("="*50)
print("PIPELINE SUMMARY")
print("="*50)

collection = vector_store._collection
embedding_count = collection.count()

print(f"✓ Documents loaded: {len(documents)}")
print(f"✓ Chunks created: {len(chunks)}")
print(f"✓ Embeddings stored: {embedding_count}")
print(f"✓ Vector store location: ../data/chroma_db")
print(f"\n✓ Pipeline is ready for semantic search!")


/var/folders/1c/1jzgd33d1ylgk1_86skny47w0000gn/T/ipykernel_63026/2291179795.py:9: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vector_store = Chroma(


Vector store loaded successfully!

Query: What is machine learning?

Retrieved 0 relevant documents:

